In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os as os
from pathlib import Path
from tysserand import tysserand as ty

In [3]:

def nodes_to_edges_unique(dossier_nodes,chemin_resultats, fichier, x_col,y_col, col_attributes):
#fonction pour enregistrer dans un dossier les edges des nodes d'un dossier 
#emplacement = path où se trouvent les nodes
#fichier = nom fichier du fichier nodes
#chemin_résultats = Path vers où on veut save les edges
#col_attributes = str = nom de la colonne contenant les attributs , exemple col_attribute = 'Cluster'
# x_col et y_col = str nom de la colonne avec les x et les y

    # Delaunay triangulation
    nodes = pd.read_parquet(dossier_nodes, columns=[x_col,y_col, col_attributes])
    coords = nodes.loc[:,[x_col,y_col]].values
    pairs = ty.build_delaunay(coords)
    distances = ty.distance_neighbors(coords, pairs)
    
    #récupération des indices comme id car delaunay reset les indices et on perd correspondance
    node_ids = nodes.index.to_numpy()
    pairs_ids = node_ids[pairs]
    edges = ty.pairs_to_df(pairs_ids)
    #enregistrer edges
    fichier = fichier.replace('nodes','edges')
    chemin = os.path.join(chemin_resultats, fichier)
    edges.to_parquet(chemin)

def nodes_to_edges_dossier(dossier_nodes,chemin_resultats, x_col,y_col, col_attributes):
    for fichier in os.listdir(dossier_nodes):
        if not (fichier.startswith("nodes") and fichier.endswith(".parquet")):
            continue
        emplacement = os.path.join(dossier_nodes, fichier)
        nodes_to_edges_unique(emplacement,chemin_resultats,fichier,x_col,y_col, col_attributes)

In [6]:
#éxécuter fonction
dossier_nodes = Path("./OUTPUT_arcsinh/states/corrected_double_status_tumour_and_macrostates_labeled/leiden_states/niches/analyses_complementaires_niches_20_merged/nodes_cluster_labeled")
chemin_resultats = Path("./OUTPUT_arcsinh/states/corrected_double_status_tumour_and_macrostates_labeled/leiden_states/niches/analyses_complementaires_niches_20_merged/edges_tysserand")
chemin_resultats.mkdir(parents= True, exist_ok= True)


nodes_to_edges_dossier(
    dossier_nodes = dossier_nodes,
    chemin_resultats = chemin_resultats,
    x_col = 'centroid-1',
    y_col = 'centroid-0',
    col_attributes = "Phenotype", )

